In [60]:
import os
import re
import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

def read(file):
    with open(file, 'r') as f:
        return f.readlines()

def fine_NPA(lines):
    start = None
    for i, line in enumerate(lines):
        if "Final electron populations and NPA charges:" in line:
            start = i + 2
            break
    if start is None:
        raise ValueError("NPA section not found in file!")

    sub_lines = lines[start:]
    end = None
    for i, line in enumerate(sub_lines):
        if line.strip() == "":
            end = i
            break
    if end is None:
        end = len(sub_lines)
    sub_lines = sub_lines[:end]

    sub_lines = sub_lines[2:]  # 헤더 2줄 스킵

    data = []
    for l in sub_lines:
        parts = re.split(r'\s+', l.strip())
        if len(parts) >= 5:
            center = parts[0]
            nuclear_charge = float(parts[1])
            electron_pop = float(parts[2])
            nmb_pop = float(parts[3])
            npa_charge = float(parts[4])
            data.append([center, nuclear_charge, electron_pop, nmb_pop, npa_charge])

    df = pd.DataFrame(data, columns=["Atom", "NuclearCharge", "ElectronPop", "NMB_Pop", "NPA_Charge"])
    return df

def find_ANGULAR(lines):
    start = None
    for i, line in enumerate(lines):
        if "Angular momentum contributions of the total atomic population:" in line:
            start = i + 2
            break
    if start is None:
        raise ValueError("Angular momentum section not found in file!")

    sub_lines = lines[start:]
    end = None
    for i, line in enumerate(sub_lines):
        if line.strip() == "":
            end = i
            break
    if end is None:
        end = len(sub_lines)
    sub_lines = sub_lines[:end]

    sub_lines = sub_lines[1:]  # 헤더 2줄 스킵

    data = []
    for l in sub_lines:
        parts = re.split(r'\s+', l.strip())
        if len(parts) >= 6:
            center = parts[0]
            s = float(parts[1]); p = float(parts[2]); d = float(parts[3]); f = float(parts[4]); g = float(parts[5])
            data.append([center, s, p, d, f, g])

    df = pd.DataFrame(data, columns=["Center", "s", "p", "d", "f", "g"])
    return df

def parse_wiberg_section(lines):
    """Wiberg-Mayer bond indices (based on density matrix in NAO basis):
    ORCA output의 'Wiberg-Mayer bond indices' 섹션을 블록 단위로 파싱하여
    완전한 결합차수 행렬(DataFrame)과 각 원자의 total valence(대각값 사전)를 반환.
    - 블록 헤더 'Centr. A/B  <col...>'를 감지
    - 각 행에서 (diag) 값이 나타나면 해당 원자의 대각 + 블록 내 col>i 항목만 매핑
    - 대각이 블록에 포함되지 않으면 해당 블록의 모든 열에 순서대로 매핑
    """
    # 1) 섹션 시작 찾기
    start = None
    for i, line in enumerate(lines):
        if "Wiberg-Mayer bond indices (based on density matrix in NAO basis):" in line:
            start = i
            break
    if start is None:
        raise ValueError("Wiberg–Mayer bond index section not found.")
    
    bond_matrix = {}
    total_valence = {}
    lines = lines[start+1 : ]
    
    for i, line in enumerate(lines):
        if line.strip() == "":
            break 
        else:
            end =  i+1 
    a = '\n'.join(lines[:end])
    return a 
def wiberg_block_to_matrix(block_text: str):
    """
    Wiberg 섹션 문자열(block_text)을 입력받아
    bond order matrix(DataFrame)와 total valence(dict)를 반환
    """
    import re
    import pandas as pd

    lines = block_text.strip().split("\n")

    # 헤더 파싱
    header_line = None
    for line in lines:
        if line.strip().startswith("Centr."):
            header_line = line
            break
    if header_line is None:
        raise ValueError("Wiberg block header not found")

    cols = re.findall(r"\d+", header_line)        # 숫자 헤더 추출
    cols = [int(c) for c in cols]

    bond_rows = {}
    total_valence = {}

    for line in lines:
        if not re.match(r"^\s*\d+", line):
            continue

        parts = re.findall(r"[-+]?\d*\.\d+|\d+", line)
        atom_idx = int(parts[0])
        values = [float(x) for x in parts[1:]]

        diag_match = re.search(r"\((.*?)\)", line)
        diag_val = float(diag_match.group(1)) if diag_match else None
        if diag_val is not None:
            total_valence[atom_idx] = diag_val
            # 첫 등장 diag 제거
            new_values, removed = [], False
            for v in values:
                if abs(v - diag_val) < 1e-8 and not removed:
                    removed = True
                    continue
                new_values.append(v)
            values = new_values
        else:
            total_valence.setdefault(atom_idx, 0.0)

        # 블록 매핑
        if atom_idx not in bond_rows:
            bond_rows[atom_idx] = {}

        if atom_idx in cols and diag_val is not None:
            bond_rows[atom_idx][atom_idx] = diag_val
            target_cols = [c for c in cols if c > atom_idx]
        else:
            target_cols = cols

        for j, v in zip(target_cols, values):
            bond_rows[atom_idx][j] = v

    all_atoms = sorted(bond_rows.keys())
    df = pd.DataFrame(index=all_atoms, columns=all_atoms, dtype=float)
    for i in all_atoms:
        for j, v in bond_rows[i].items():
            df.loc[i, j] = v

    sym = df.copy()
    for i in df.index:
        for j in df.columns:
            if pd.isna(df.loc[i,j]) and not pd.isna(df.loc[j,i]):
                sym.loc[i,j] = df.loc[j,i]
            elif pd.isna(df.loc[j,i]) and not pd.isna(df.loc[i,j]):
                sym.loc[j,i] = df.loc[i,j]
    return sym, total_valence
def compute_sp_hybrid(df_angular):
    df = df_angular.copy()
    df["p/s"] = df["p"] / df["s"]
    df["hybrid_type"] = df["p/s"].apply(assign_hybrid_label)
    return df
def assign_hybrid_label(ratio):
    if ratio < 1.2:
        return "sp"         # 직선형
    elif ratio < 2.2:
        return "sp2"        # 평면형
    elif ratio < 3.5:
        return "sp3"        # 사면체형
    else:
        return "sp? (high p)"  # 고전적 혼성 불명확(고전적 sp 계열 아님)


# =========== 사용 예시 ===========
TFSI_JANPA = read("./00_2_orca/VBBI+/input.janpa.output")
# TFSI_JANPA = read("./00_2_orca/TFSI-/input.janpa.output")
df_npa = fine_NPA(TFSI_JANPA)
df_angular = find_ANGULAR(TFSI_JANPA)
bondsection = parse_wiberg_section(TFSI_JANPA)
df, total_valence = wiberg_block_to_matrix(bondsection)
df_angular_result = compute_sp_hybrid(df_angular)
print(df_angular_result[["Center", "s", "p", "p/s", "hybrid_type"]])


   Center         s         p       p/s hybrid_type
0      C1  3.057858  3.293826  1.077168          sp
1      C2  2.956315  3.284391  1.110975          sp
2      H3  0.770651  0.001151  0.001494          sp
3      H4  0.794307  0.001150  0.001448          sp
4      C5  2.870920  3.187386  1.110231          sp
5      H6  0.767280  0.001142  0.001489          sp
6      C7  2.939820  3.245891  1.104112          sp
7      C8  2.939243  3.272226  1.113289          sp
8      C9  2.860950  3.233468  1.130208          sp
9     C10  2.941229  3.252390  1.105793          sp
10    C11  2.938514  3.284007  1.117574          sp
11    H12  0.756641  0.001131  0.001495          sp
12    H13  0.767456  0.001139  0.001485          sp
13    H14  0.758894  0.001157  0.001525          sp
14    H15  0.758591  0.001119  0.001476          sp
15    C16  2.995914  3.252118  1.085518          sp
16    N17  3.234439  4.270196  1.320228         sp2
17    H18  0.794795  0.001046  0.001317          sp
18    H19  0

In [ ]:
'''(MOL2) --> parse_mol2_bonds --> bond list
                               ↓
                       build_bond_graph
                               ↓
                             adj
             ┌───────────────┴───────────────┐
   generate_angles(adj)            generate_dihedrals(adj)'''


In [69]:
import logging
from collections import defaultdict
from itertools import combinations

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')


def parse_mol2_bonds(mol2_file):
    bonds = []
    in_bond_section = False

    with open(mol2_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("@<TRIPOS>BOND"):
                in_bond_section = True
                continue
            if line.startswith("@<TRIPOS>") and in_bond_section:
                break
            if in_bond_section and line:
                parts = line.split()
                if len(parts) >= 4:
                    a = int(parts[1]); b = int(parts[2])
                    bond = (a, b) if a < b else (b, a)
                    bonds.append(bond)

    bonds = sorted(set(bonds))
    logging.info(f"Parsed {len(bonds)} bonds from {mol2_file}")
    return bonds


def build_bond_graph(bonds):
    adj = defaultdict(set)
    for a, b in bonds:
        adj[a].add(b)
        adj[b].add(a)
    adj_sorted = {node: sorted(neigh) for node, neigh in adj.items()}
    logging.info(f"Built bond graph with {len(adj_sorted)} atoms")
    return adj_sorted


def generate_angles(adj):
    angle_set = set()
    for j, neighbors in adj.items():
        if len(neighbors) < 2:
            continue
        for i, k in combinations(neighbors, 2):
            angle = (i, j, k) if i < k else (k, j, i)
            angle_set.add(angle)
    angles = sorted(angle_set, key=lambda x: (x[1], x[0], x[2]))
    logging.info(f"Generated {len(angles)} angles")
    return angles


def generate_dihedrals(adj):
    """
    dihedral (i, j, k, l) 생성
    j-k가 중심 결합. i는 j의 다른 이웃, l은 k의 다른 이웃.
    """
    dihedral_set = set()
    for j in adj:
        for k in adj[j]:
            if j < k:  # 중심 결합 방향 정렬
                for i in adj[j]:
                    if i == k:
                        continue
                    for l in adj[k]:
                        if l == j:
                            continue
                        if i != l:  # 순환 피함
                            dihedral = (i, j, k, l)
                            dihedral_set.add(dihedral)
    dihedrals = sorted(dihedral_set, key=lambda x: (x[1], x[2], x[0], x[3]))
    logging.info(f"Generated {len(dihedrals)} dihedrals")
    return dihedrals


# ===== 실행 =====
mol2_path = "./00_2_orca/TFSI-/input.mol2"
bonds = parse_mol2_bonds(mol2_path)
adj = build_bond_graph(bonds)

print("\nBond List:")
print(bonds)

print("\nBond Graph (Adjacency):")
for atom, neighbors in adj.items():
    print(f"{atom}: {neighbors}")

angles = generate_angles(adj)
print("\nAngle List (i, j, k):")
for a in angles:
    print(a)

dihedrals = generate_dihedrals(adj)
print("\nDihedral List (i, j, k, l):")
for d in dihedrals:
    print(d)


INFO: Parsed 14 bonds from ./00_2_orca/TFSI-/input.mol2
INFO: Built bond graph with 15 atoms
INFO: Generated 25 angles
INFO: Generated 24 dihedrals



Bond List:
[(1, 3), (1, 4), (1, 5), (1, 9), (2, 6), (2, 7), (2, 8), (2, 10), (9, 11), (9, 12), (9, 13), (10, 11), (10, 14), (10, 15)]

Bond Graph (Adjacency):
1: [3, 4, 5, 9]
3: [1]
4: [1]
5: [1]
9: [1, 11, 12, 13]
2: [6, 7, 8, 10]
6: [2]
7: [2]
8: [2]
10: [2, 11, 14, 15]
11: [9, 10]
12: [9]
13: [9]
14: [10]
15: [10]

Angle List (i, j, k):
(3, 1, 4)
(3, 1, 5)
(3, 1, 9)
(4, 1, 5)
(4, 1, 9)
(5, 1, 9)
(6, 2, 7)
(6, 2, 8)
(6, 2, 10)
(7, 2, 8)
(7, 2, 10)
(8, 2, 10)
(1, 9, 11)
(1, 9, 12)
(1, 9, 13)
(11, 9, 12)
(11, 9, 13)
(12, 9, 13)
(2, 10, 11)
(2, 10, 14)
(2, 10, 15)
(11, 10, 14)
(11, 10, 15)
(14, 10, 15)
(9, 11, 10)

Dihedral List (i, j, k, l):
(3, 1, 9, 11)
(3, 1, 9, 12)
(3, 1, 9, 13)
(4, 1, 9, 11)
(4, 1, 9, 12)
(4, 1, 9, 13)
(5, 1, 9, 11)
(5, 1, 9, 12)
(5, 1, 9, 13)
(6, 2, 10, 11)
(6, 2, 10, 14)
(6, 2, 10, 15)
(7, 2, 10, 11)
(7, 2, 10, 14)
(7, 2, 10, 15)
(8, 2, 10, 11)
(8, 2, 10, 14)
(8, 2, 10, 15)
(1, 9, 11, 10)
(12, 9, 11, 10)
(13, 9, 11, 10)
(2, 10, 11, 9)
(14, 10, 11, 9)
(15, 10, 1

In [67]:
angles

[(7, 2, 10),
 (5, 1, 9),
 (2, 10, 15),
 (3, 1, 5),
 (12, 9, 13),
 (4, 1, 5),
 (14, 10, 15),
 (1, 9, 13),
 (9, 11, 10),
 (4, 1, 9),
 (2, 10, 11),
 (2, 10, 14),
 (11, 10, 15),
 (3, 1, 4),
 (11, 9, 13),
 (6, 2, 8),
 (1, 9, 11),
 (8, 2, 10),
 (1, 9, 12),
 (7, 2, 8),
 (11, 10, 14),
 (3, 1, 9),
 (6, 2, 7),
 (6, 2, 10),
 (11, 9, 12)]